# Tutorial 1: Run `dkx`, Write Outputs, And Plot Diagnostics

This notebook is the fastest path from an `input.namelist` to useful output files. It uses a tiny teaching input so the workflow is about file formats, diagnostics, and interpretation rather than production runtime.

## What The Run Solves

`dkx` solves a radially local drift-kinetic equation for the first-order distribution correction $f_{s1}$. The output files also store flux-surface averages, transport diagnostics, and solver metadata so a run can be audited after it finishes. In this tutorial we skip the expensive solve and write the standard output schema, which is useful for learning the I/O and plotting tools.

In [ ]:
from pathlib import Path

repo = Path.cwd()
input_path = repo / "examples" / "getting_started" / "input.namelist"
out_dir = repo / "tutorial_output"
out_dir.mkdir(exist_ok=True)
input_path

## One Command For New Users

The helper script writes HDF5, NetCDF, and NPZ files and then creates a PDF diagnostics panel. It is equivalent to using the CLI plus `dkx --plot`, but keeps the tutorial output together in one folder.

In [ ]:
!python examples/tutorials/run_quick_output_and_plot.py --out-dir tutorial_output

## Inspect The Output Schema

A production run writes the same schema, but with solved distribution-function and transport fields populated. The fields shown here are the stable file contract used by plotting, parity checks, and downstream workflows.

In [ ]:
from dkx.io import read_sfincs_output_file

data = read_sfincs_output_file(out_dir / "sfincsOutput_tutorial.h5")
for key in sorted(data)[:20]:
    value = data[key]
    shape = getattr(value, "shape", ())
    print(f"{key:40s} shape={shape}")

## Plot The Diagnostics Panel

The plotter reads any supported output file and writes a PDF or image. For solved runs, the panel includes the quantities needed to inspect fluxes, flows, bootstrap current, solver status, and output consistency.

In [ ]:
from dkx.plotting import plot_sfincs_output_summary

panel = out_dir / "sfincsOutput_tutorial_summary_from_notebook.pdf"
plot_sfincs_output_summary(input_h5=out_dir / "sfincsOutput_tutorial.h5", output_png=panel)
panel

## Next Steps

Run a real solve by removing the schema-only shortcut and calling the CLI directly:

```bash
dkx examples/sfincs_examples/quick_2species_FPCollisions_noEr/input.namelist --out sfincsOutput.h5
dkx --plot sfincsOutput.h5
```

For VMEC geometry, add `--wout-path /path/to/wout.nc` or put the VMEC path in the namelist.